# Part F - RAG metrics

## Get LLM responses

In [ ]:
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer
import json
from pathlib import Path
from datetime import datetime
import json
from pathlib import Path
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.settings import ModelSettings
import nest_asyncio
nest_asyncio.apply() # To be able to run pydantic in jupyter notebooks preventing event loop issues



database_name = "tesis_MCD_v1"
collection_name = "codigo_de_mineria_v1"
embedding_model_name = "hiiamsid/sentence_similarity_spanish_es" # https://huggingface.co/hiiamsid/sentence_similarity_spanish_es/tree/main
embedding_model = SentenceTransformer(embedding_model_name)
answer_vector_field = "vector_contenido_texto"
output_fields = [
    "codigo_norma",
    "id_parte",
    "id_padre",
    "tipo_parte",
    "numero_parte",
    "titulo_parte",
    "ultimo_estado",
    "contenido_texto",
]

def get_db_client():
    client = MilvusClient(
        uri="http://localhost:19530",
        token="root:Milvus"
    )
    client.use_database(database_name)
    return client

def get_matches(text:str) -> list[dict]:
    try:
        client = get_db_client()
        question_vector = embedding_model.encode(text)

        results = client.search(
            collection_name=collection_name,
            anns_field=answer_vector_field,
            data=[question_vector],
            limit=5000,
            search_params={"metric_type": "COSINE"},
            output_fields=output_fields
        )
    except Exception as e:
        print(e)
    finally:
        client.close()

    return [{"index": i, "distance": result["distance"], **result["entity"]} for i, result in enumerate(results[0], start=1)]

def generate_llm_responses(question, model, system_prompt, model_settings, tools=None, rag_data=None) -> dict:
    agent = Agent(
        model=model,
        system_prompt=system_prompt,
        model_settings=model_settings
        )
    agent.model_settings

    result = agent.run_sync(
        question,
        model_settings=model_settings,
    )

    respuesta = {
        "texto": result.output,
        "metadata": {# response metadata
            "model_settings": agent.model_settings,
            "model_version": result.response.model_name,
            "provider_name": result.response.provider_name,
            "timestamp": result.timestamp().isoformat(),
            "rag_data_used": rag_data if rag_data else "",
            "usage": {
                "tokens": {
                    "input_tokens": result.usage().input_tokens,
                    "output_tokens": result.usage().output_tokens,
                    "tokens_details": result.usage().details,
                },
                "cost": {
                    "input_price": float(result.response.cost().input_price),
                    "output_price": float(result.response.cost().output_price),
                    "total_price": float(result.response.cost().total_price),
                    "model_input_price_mtok": float(result.response.cost().model_price.input_mtok),
                    "model_output_price_mtok": float(result.response.cost().model_price.output_mtok),
                    "model_id": result.response.cost().model.id,
                    "model_name": result.response.cost().model.name,
                },
            },
        }
    }
    print(result)

    return respuesta



In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_PATH = Path.cwd()
DATA_FILE_PATH = BASE_PATH.joinpath("20251101_100000_results.json")
OUTPUT_FILE_PATH = BASE_PATH.joinpath(f"{timestamp}_results.json")


with open(DATA_FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

In [27]:
tools = []

model_settings = ModelSettings(
    temperature=0.0, # As deterministic responses as possible
    # top_p=0.1,
)

system_prompt = """
Eres un asistente legal experto en derecho Argentino. Respondes preguntas legales basadas en la Constitución Nacional Argentina, Códigos, Leyes y jurisprudencia. Siempre que sea posible citas las fuentes legales en tu respuesta. Si no sabes la respuesta, dices que no lo sabes.
Dentro del texto de la respuesta debe mencionarse de donde provino la afirmación. Por ejemplo "según el artículo... de la ...". Las respuestas deben ser claras, concisas y directas al punto.
Solo retorna la respuesta.
Tienes a tu disposición la siguiente información obtenida de un sistema RAG (Retrieval Augmented Generation). Utiliza esta información para responder la pregunta de la mejor manera posible.
=========
{retrieved_data}
=========

"""

models=[
    "openai:gpt-5-nano",
    "openai:gpt-5-mini",
    "openai:gpt-5",
]

for pregunta in data:
    retrieved_data = get_matches(pregunta["texto"])[:10] # Get top 10 matches
    selected_retrieved_data = [{k: v for k, v in entry.items() if k in ["codigo_norma", "tipo_parte", "numero_parte", "ultimo_estado", "contenido_texto", "index", "distance"]} for entry in retrieved_data]
    for model in models:
        # Check if response for this model already exists
        # This checks does not account for different system prompts
        if any(
                model.endswith(resp["metadata"]["usage"]["cost"]["model_id"])
                and resp["metadata"]["model_settings"] == model_settings
                for resp in pregunta.get("llm_responses_base_rag", [])
            ):
            print(f"Response for model '{model}', question '{pregunta['numero_pregunta']}', subpregunta '{pregunta.get('numero_subpregunta', None)}', and model settings already exists. Skipping...")
            continue

        if "llm_responses_base_rag" not in pregunta:
            pregunta["llm_responses_base_rag"] = []

        print(f"Generating response with model '{model}', question '{pregunta['numero_pregunta']}' and subpregunta '{pregunta.get('numero_subpregunta', None)}'")
        pregunta["llm_responses_base_rag"].append(generate_llm_responses(pregunta["texto"], model, system_prompt.format(retrieved_data=selected_retrieved_data), model_settings, rag_data=retrieved_data))

        with open(OUTPUT_FILE_PATH, encoding="utf-8", mode="w") as f:
            f.write(json.dumps(data, ensure_ascii=False, indent=4))


Response for model 'openai:gpt-5-nano', question '1', subpregunta '1', and model settings already exists. Skipping...
Response for model 'openai:gpt-5-mini', question '1', subpregunta '1', and model settings already exists. Skipping...
Response for model 'openai:gpt-5', question '1', subpregunta '1', and model settings already exists. Skipping...
Generating response with model 'openai:gpt-5-nano', question '1' and subpregunta '2'
AgentRunResult(output='Sí. El Estado puede otorgar la concesión de explotación a Juan Pérez incluso frente a la oposición de la propietaria de la finca, conforme al Código de Minería Argentino, porque la explotación de los minerales no depende de la voluntad del dueño del suelo y el derecho minero nace de la autoridad minera.\n\n- Derecho a permisos y concesiones: “Toda persona física o jurídica puede solicitar de la autoridad permisos exclusivos para explorar un área determinada… Los titulares de permisos de exploración tendrán el derecho exclusivo a obtener 

In [23]:
retrieved_data

[{'index': 1,
  'distance': 0.6904841065406799,
  'numero_parte': 25,
  'titulo_parte': None,
  'ultimo_estado': 'en vigor',
  'contenido_texto': 'Toda persona física o jurídica puede solicitar de la autoridad permisos exclusivos para explorar un área determinada, por el tiempo y en la extensión que señala la ley.\n\nLos titulares de permisos de exploración tendrán el derecho exclusivo a obtener concesiones de explotación dentro de las áreas correspondientes a los permisos.\n\nPara obtener el permiso se presentará una solicitud que consigne las coordenadas de los vértices del área solicitada y que exprese el objeto de esa exploración, el nombre y domicilio del solicitante y del propietario del terreno.\n\nLa solicitud contendrá también el programa mínimo de trabajos a realizar, con una estimación de las inversiones que proyecta efectuar e indicación de los elementos y equipos a utilizar.\n\nIncluirá también una declaración jurada sobre la inexistencia de las prohibiciones resultantes d